# Video Studio — Colab + Cloudflare Tunnel

Gemma2:2b runs through Ollama in Colab. Edge TTS provides narration and FFmpeg renders the MP4.

Select a T4 GPU runtime and run Cells 1–6 in order. In Cell 3, upload the updated `video_api_source.zip`. Cell 4 downloads the model on its first run.

Open the Video Studio link printed by Cell 6, paste its `api_token`, and connect. Choose files from your computer in the browser, then create, track, and download videos there.

Cell 7 provides the optional Postman collection and source archive. Cells 8 and 9 inspect or stop the service. Quick Tunnel URLs can change when recreated, and outputs live on the Colab runtime. Download results before the runtime ends.


## Cell 1 — Install dependencies

Run once per fresh runtime. FFmpeg/FFprobe and fonts are installed only if missing.

In [ ]:
import subprocess
import sys
import shutil
from pathlib import Path

DEPENDENCIES = ['fastapi>=0.115,<1', 'starlette>=0.46,<2', 'uvicorn>=0.30,<1', 'pydantic>=2.10,<3', 'python-multipart>=0.0.20,<1', 'Pillow>=10.4,<13', 'numpy>=1.26,<3', 'edge-tts>=7.3,<8', 'jsonschema>=4.23,<5', 'imageio-ffmpeg>=0.5,<1', 'requests>=2.32,<3', 'zstandard>=0.23,<1', 'opencv-python-headless>=4.10,<5', 'ipython>=8,<10']
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *DEPENDENCIES], check=True)

missing_ffmpeg = not shutil.which("ffmpeg") or not shutil.which("ffprobe")
font_roots = [Path("/usr/share/fonts/truetype/dejavu"), Path("/usr/share/fonts/truetype/liberation2")]
missing_fonts = not any(path.exists() for path in font_roots)
if missing_ffmpeg or missing_fonts:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg", "fonts-dejavu-core"], check=True)

print("Dependencies ready. Continue with Cell 2.")


## Cell 2 — Service settings

Keep the defaults for a Quick Tunnel; it needs no Cloudflare account or domain. The access token
for **your own API** is generated automatically. It is separate from any provider key.

Optional named tunnel: supply its connector token and existing public hostname. Its Cloudflare
application route must point to the local port below. Do not prepare the model while a render is active.

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/savvy_video_api")
API_PORT = 8000

# Optional: leave BOTH empty for the default temporary Quick Tunnel.
CLOUDFLARE_TUNNEL_TOKEN = ""
PUBLIC_BASE_URL = ""  # Named-tunnel example: https://video.your-domain.com

APP_DIR = BASE_DIR / "app"
STATE_DIR = BASE_DIR / "state"
BASE_DIR.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(exist_ok=True)

if not 1024 <= API_PORT <= 65535:
    raise ValueError("Choose an API_PORT between 1024 and 65535.")
print("Application:", APP_DIR)
print("Job storage:", STATE_DIR / "jobs")
print("Set prompt, voice, FPS, dimensions and media in Postman, not in this cell.")


## Cell 3 — Upload `video_api_source.zip`

Run this cell, select **Choose Files**, and pick the updated **video_api_source.zip** from your computer. Upload the ZIP as-is. The full project ZIP is a separate download.

When the cell prints **Video Studio files ready**, continue to Cell 4.


In [ ]:
# CELL 3 — Upload and unpack the application ZIP
from google.colab import files
import importlib
import io
import sys
import zipfile

print("Choose the updated video_api_source.zip from your computer.")
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Choose exactly one file: the updated video_api_source.zip.")

zip_bytes = next(iter(uploaded.values()))
try:
    application_zip = zipfile.ZipFile(io.BytesIO(zip_bytes))
except zipfile.BadZipFile as exc:
    raise ValueError("Choose the updated video_api_source.zip. The selected file is not a readable ZIP.") from exc

with application_zip as archive:
    required = {
        "server.py", "runner.py", "launch.py", "models.py",
        "media.py", "store.py", "duration.py",
        "studio/index.html", "studio/studio.css", "studio/studio.js",
        "studio/storyboard-example.json",
        "pipeline_stages.json", "example_config.json",
        "default_logo_transparent.png", "default_closing_slide.png",
        "Savvy_Video_API.postman_collection.json",
    }

    missing = sorted(required - set(archive.namelist()))
    if missing:
        raise ValueError(
            "Choose the updated video_api_source.zip from the project. "
            "This archive is missing: " + ", ".join(missing)
        )

    for entry in archive.infolist():
        destination = (APP_DIR / entry.filename).resolve()
        if not destination.is_relative_to(APP_DIR.resolve()):
            raise ValueError("Unexpected path inside the ZIP.")

    archive.extractall(APP_DIR)

SOURCE_ZIP = BASE_DIR / "video_api_source.zip"
SOURCE_ZIP.write_bytes(zip_bytes)

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

importlib.invalidate_caches()
print("Video Studio files ready:", APP_DIR)
print("Continue to Cell 4.")


## Cell 4 — Prepare local Gemma2:2b

First run downloads the official Ollama runtime and model, which can take several minutes.
Wait until this cell finishes. Later runs reuse them within the same Colab session.
The same setup runs automatically if you skip this cell, but the first job then takes longer.

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-u", str(APP_DIR / "runner.py"), "--prepare", str(STATE_DIR)],
    cwd=str(APP_DIR),
    check=True,
)


## Cell 5 — Start the API and render worker

One render runs at a time. Each job has its own process, files and log.
This cell returns after the API is ready; continue to Cell 6 for the public Postman address.

In [ ]:
import importlib
import launch

importlib.reload(launch)
API_CONNECTION = launch.start_api(STATE_DIR, API_PORT)
API_TOKEN = API_CONNECTION["api_token"]
LOCAL_URL = API_CONNECTION["local_url"]
print("API is ready inside Colab:", LOCAL_URL)
print("Continue to Cell 6 for the public URL to use in Postman.")


## Cell 6 — Open Video Studio through Cloudflare

Open the printed Video Studio link. Its Cloudflare base URL is filled in automatically. Paste the printed `api_token` into the Colab API token field and select Connect. If you open Studio from a local server, paste `base_url` too.

Keep the API token private. Quick Tunnel URLs change when recreated.


In [ ]:
PUBLIC_URL = launch.start_tunnel(
    STATE_DIR,
    port=API_PORT,
    tunnel_token=CLOUDFLARE_TUNNEL_TOKEN,
    public_base_url=PUBLIC_BASE_URL,
)

print("Video Studio:", PUBLIC_URL.rstrip("/") + "/studio/")
print("base_url =", PUBLIC_URL)
print("api_token =", API_TOKEN)
print("Open Video Studio, paste api_token, and select Connect.")
print("API documentation:", PUBLIC_URL.rstrip("/") + "/docs")


## Cell 7 — Download optional Postman tools

Import the downloaded collection into Postman. Set its two variables from Cell 6.
**Create video — prompt only** is preconfigured as a faster 960x540 test.
The create request automatically saves `job_id` for status/report/download requests.

Set DOWNLOAD_SOURCE_ZIP=True if you also want the full Python project and README.
**Create video — supplied storyboard** accepts a JSON narration plan and includes File rows for a company logo, background video, scene media, and closing artwork. Replace `config.storyboard` with the larger LLM output. For any upload you skip, uncheck its File row and remove its reference from `config`. The example and reusable prompt are in the application ZIP.


In [ ]:
from google.colab import files

DOWNLOAD_SOURCE_ZIP = False
files.download(str(APP_DIR / "Savvy_Video_API.postman_collection.json"))
if DOWNLOAD_SOURCE_ZIP:
    files.download(str(SOURCE_ZIP))
print("Detailed instructions:", APP_DIR / "README.md")


## Cell 8 — Monitor jobs or inspect the latest error

Run whenever you want a local status summary. In Postman the same information is available from
GET /jobs and GET /jobs/{job_id}. The report shows requested/actual duration and any extension.

In [ ]:
import requests

session = requests.Session()
session.trust_env = False
headers = {"Authorization": "Bearer " + API_TOKEN}
response = session.get(LOCAL_URL + "/jobs", headers=headers, timeout=15)
response.raise_for_status()
for job in response.json():
    print(job["job_id"], "|", job["status"], "|", job["stage"])
    if job["result"]:
        print("  Requested:", job["result"]["requested_seconds"],
              "Actual:", job["result"]["actual_seconds"], "seconds")
    if job["error"]:
        print("  Error:", job["error"])
        print("  Log:", PUBLIC_URL + "/jobs/" + job["job_id"] + "/log")


## Cell 9 — Stop the API and tunnel

Run after your jobs finish and videos have been downloaded. Stopping during rendering interrupts
the active job. Files remain in this runtime's state folder until the runtime storage is deleted.

In [ ]:
launch.stop_services(STATE_DIR)
